# Задачи с сайта leetcode.com


In [1]:
import sqlite3

import pandas as pd

con = sqlite3.connect("sqlite.db")
cur = con.cursor()

In [2]:
def execute_formatted_sqlite(sql_schema):
    sql = ";\n".join(sql_schema.split("\n"))
    cur.executescript(sql)
    con.commit()


def select(sql):
    return pd.read_sql(sql, con)

### 175. Combine Two Tables

Write a solution to report the first name, last name, city, and state of each person in the Person table. If the address of a personId is not present in the Address table, report null instead.

Return the result table in any order.


In [3]:
# SQLite3 не знает что такое TRUNCATE TABLE, поэтому нужно из запроса удалить эти команды.
# Добавить PRIMARY KEY к полям Id и OR REPLACE или IGNORE к INSERT, чтобы не дублировались строки при перезапуске.

sql_schema = """
Create table If Not Exists Person (personId int primary key, firstName varchar(255), lastName varchar(255))
Create table If Not Exists Address (addressId int primary key, personId int, city varchar(255), state varchar(255))
insert or replace into Person (personId, lastName, firstName) values ('1', 'Wang', 'Allen')
insert or replace into Person (personId, lastName, firstName) values ('2', 'Alice', 'Bob')
insert or replace into Address (addressId, personId, city, state) values ('1', '2', 'New York City', 'New York')
insert or replace into Address (addressId, personId, city, state) values ('2', '3', 'Leetcode', 'California')"""

# А так же после кажной строки поставить ';'
sql = ";\n".join(sql_schema.split("\n"))
print(sql)

# Так как несколько команд, нужен executescript(), а не execute()
cur.executescript(sql)

# В SQLite3 в Python нет автокомита (в CLI есть), поэтому:
con.commit()

;
Create table If Not Exists Person (personId int primary key, firstName varchar(255), lastName varchar(255));
Create table If Not Exists Address (addressId int primary key, personId int, city varchar(255), state varchar(255));
insert or replace into Person (personId, lastName, firstName) values ('1', 'Wang', 'Allen');
insert or replace into Person (personId, lastName, firstName) values ('2', 'Alice', 'Bob');
insert or replace into Address (addressId, personId, city, state) values ('1', '2', 'New York City', 'New York');
insert or replace into Address (addressId, personId, city, state) values ('2', '3', 'Leetcode', 'California')


In [4]:
sql = """--sql
select
    p.firstName,
    p.lastName,
    a.city,
    a.state
from
    person p
    left join Address a on p.personid = a.personid"""
select(sql)

,firstName,lastName,city,state
0,Allen,Wang,None,None
1,Bob,Alice,New York City,New York


### 176. Second Highest Salary

Write a solution to find the second highest distinct salary from the Employee table. If there is no second highest salary, return null (return None in Pandas).


In [5]:
sql_schema = """
Create table If Not Exists Employee (id int primary key, salary int)
insert or replace into Employee (id, salary) values ('1', '100')
insert or replace into Employee (id, salary) values ('2', '200')
insert or replace into Employee (id, salary) values ('3', '300')"""
execute_formatted_sqlite(sql_schema)

In [6]:
# Если в таблице всего одна зарплата, OFFSET 1 не находит строк — запрос не возвращает вообще ничего.
# Вложив запрос в SELECT (...) — он вернёт строку, даже если внутри ничего не найдено.

sql = """--sql
select
    (
        select distinct
            e.salary
        from
            employee e
        order by
            e.salary desc
        limit
            1
        offset
            1
    ) as second_highest_salary"""
select(sql)

,second_highest_salary
0,200


### 177. Nth Highest Salary

Write a solution to find the nth highest distinct salary from the Employee table. If there are less than n distinct salaries, return null.


In [7]:
sql_schema = """
Create table If Not Exists Employee (Id int primary key, Salary int)
insert or replace into Employee (id, salary) values ('1', '100')
insert or replace into Employee (id, salary) values ('2', '200')
insert or replace into Employee (id, salary) values ('3', '300')"""
execute_formatted_sqlite(sql_schema)